# Kaggle Train With Fast Git Sync

Notebook này dùng GPU Kaggle để train Tiny YOLO from scratch. Nếu repo đã có trong `/kaggle/working/XLA` thì chỉ đồng bộ code mới từ GitHub; nếu chưa có thì clone lần đầu.

In [ ]:
REPO_URL = "https://github.com/huyvanzzz/XLA.git"
BRANCH = "main"
WORK_DIR = "/kaggle/working/XLA"

# Sửa đường dẫn này theo Kaggle Dataset của bạn.
KAGGLE_PUBLIC_DIR = "/kaggle/input/xla-object-detection/public"

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

os.chdir("/kaggle/working")
work_path = Path(WORK_DIR)

if (work_path / ".git").exists():
    print("Repo exists, syncing latest code...")
    subprocess.run(["git", "-C", WORK_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    if work_path.exists():
        shutil.rmtree(work_path)
    print("Repo not found, cloning...")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK_DIR], check=True)

os.chdir(WORK_DIR)
print("Using repo at", WORK_DIR)
!git log --oneline -1

In [ ]:
# Kaggle có thể dùng PyTorch quá mới không hỗ trợ Tesla P100 (sm_60).
# Bản này tương thích P100 và vẫn chạy tốt trên GPU Kaggle phổ biến.
!python -m pip uninstall -y -q torch torchvision torchaudio
!python -m pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 torch==2.4.1+cu121
!python -m pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import shutil

src_public = Path(KAGGLE_PUBLIC_DIR)
dst_public = Path(WORK_DIR) / "public"

if src_public.exists():
    if dst_public.exists():
        shutil.rmtree(dst_public)
    shutil.copytree(src_public, dst_public)
    print("Copied dataset from", src_public)
elif dst_public.exists():
    print("Using existing public/ inside repo")
else:
    raise FileNotFoundError(f"Cannot find dataset at {src_public}. Upload public/ as a Kaggle Dataset and update KAGGLE_PUBLIC_DIR.")

In [ ]:
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
!python train.py \
  --train_data ./public/annotations/train.json \
  --val_data ./public/annotations/val.json \
  --image_dir ./public/train/images \
  --val_image_dir ./public/val/images \
  --checkpoint_dir ./models/ \
  --config ./configs/default.yaml

In [ ]:
!python predict.py \
  --image_dir ./public/val/images \
  --output ./val_predictions.json \
  --checkpoint ./models/best.pth \
  --config ./configs/default.yaml \
  --batch_size 16

!python public/tools/evaluate_predictions.py \
  --ground_truth ./public/annotations/val.json \
  --predictions ./val_predictions.json \
  --output ./val_score.json

!cat ./val_score.json

In [ ]:
# Checkpoint inference sweep: chay sau khi da co ./models/best.pth.
# Muc tieu: test ranking/inference tren checkpoint hien tai, khong train lai.
import csv
import json
import subprocess
from pathlib import Path

FAST_SWEEP = True  # True: it hon de xem nhanh; False: full sweep lau hon.

checkpoint = Path("./models/best.pth")
if not checkpoint.exists():
    raise FileNotFoundError("Missing ./models/best.pth. Train first or upload/copy best.pth into ./models/.")

sweep_dir = Path("./models/checkpoint_sweep")
sweep_dir.mkdir(parents=True, exist_ok=True)

# DQP = Distribution Quality Power. 0.0 la baseline checkpoint cu.
# TTA=True cham hon nhung gan voi final scoring hon.
experiments = []
dqp_values = [0.0, 0.20, 0.25, 0.35] if FAST_SWEEP else [0.0, 0.10, 0.20, 0.25, 0.35, 0.50]
for dqp in dqp_values:
    experiments.append({"name": f"dqp{dqp:.2f}_tta0", "dqp": dqp, "tta": False, "conf": None, "nms": None})
for dqp in dqp_values:
    experiments.append({"name": f"dqp{dqp:.2f}_tta1", "dqp": dqp, "tta": True, "conf": None, "nms": None})

# Nho hon, nhung hay co ich neu best dang bi nhieu duplicate/FP.
conf_values = [0.01, 0.02, 0.04] if FAST_SWEEP else [0.005, 0.01, 0.02, 0.04, 0.08]
for conf in conf_values:
    experiments.append({"name": f"conf{conf:g}_dqp025_tta1", "dqp": 0.25, "tta": True, "conf": conf, "nms": None})
nms_values = [0.45, 0.50, 0.55] if FAST_SWEEP else [0.45, 0.50, 0.55, 0.60]
for nms in nms_values:
    experiments.append({"name": f"nms{nms:.2f}_dqp025_tta1", "dqp": 0.25, "tta": True, "conf": 0.01, "nms": nms})

results = []
for exp in experiments:
    pred_path = sweep_dir / f"{exp['name']}_predictions.json"
    score_path = sweep_dir / f"{exp['name']}_score.json"
    cmd = [
        "python", "predict.py",
        "--image_dir", "./public/val/images",
        "--output", str(pred_path),
        "--checkpoint", str(checkpoint),
        "--config", "./configs/default.yaml",
        "--batch_size", "32",
        "--distribution_quality_power", str(exp["dqp"]),
    ]
    # predict.py lay tta_hflip tu config; override nhanh bang config tam se phuc tap.
    # Neu muon tat TTA, dung env patch tam qua file config clone.
    config_path = Path("./configs/default.yaml")
    temp_config = None
    if not exp["tta"]:
        text = config_path.read_text(encoding="utf-8")
        text = text.replace("  tta_hflip: true", "  tta_hflip: false")
        temp_config = sweep_dir / f"{exp['name']}_config.yaml"
        temp_config.write_text(text, encoding="utf-8")
        cmd[cmd.index("./configs/default.yaml")] = str(temp_config)
    if exp["conf"] is not None:
        cmd += ["--conf_threshold", str(exp["conf"])]
    if exp["nms"] is not None:
        cmd += ["--nms_threshold", str(exp["nms"])]

    print("\nRUN", exp["name"], "dqp=", exp["dqp"], "tta=", exp["tta"], "conf=", exp["conf"], "nms=", exp["nms"])
    subprocess.run(cmd, check=True)
    subprocess.run([
        "python", "public/tools/evaluate_predictions.py",
        "--ground_truth", "./public/annotations/val.json",
        "--predictions", str(pred_path),
        "--output", str(score_path),
    ], check=True)
    score = json.loads(score_path.read_text(encoding="utf-8"))
    row = {
        "name": exp["name"],
        "dqp": exp["dqp"],
        "tta": exp["tta"],
        "conf": exp["conf"] if exp["conf"] is not None else "ckpt/config",
        "nms": exp["nms"] if exp["nms"] is not None else "ckpt/config",
        "map50": score.get("map50", score.get("mAP", score.get("map"))),
        "precision": score.get("precision"),
        "recall": score.get("recall"),
        "score_path": str(score_path),
        "pred_path": str(pred_path),
    }
    per_class = score.get("per_class", {})
    for cls_name, cls_score in per_class.items():
        if isinstance(cls_score, dict):
            row[f"ap_{cls_name}"] = cls_score.get("ap")
    results.append(row)
    print(json.dumps(row, ensure_ascii=False, indent=2))

summary_json = sweep_dir / "summary.json"
summary_csv = sweep_dir / "summary.csv"
summary_json.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
with summary_csv.open("w", newline="", encoding="utf-8") as f:
    fieldnames = sorted({key for row in results for key in row.keys()})
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

ranked = sorted(results, key=lambda r: (r["map50"] if r["map50"] is not None else -1), reverse=True)
print("\nTOP RESULTS")
for row in ranked[:10]:
    print(json.dumps(row, ensure_ascii=False))
print("Saved", summary_json, summary_csv)


In [ ]:
from pathlib import Path
import shutil

artifact_dir = Path("/kaggle/working/artifacts")
artifact_dir.mkdir(exist_ok=True)
for name in ["best.pth", "last.pth"]:
    src = Path("./models") / name
    if src.exists():
        shutil.copy2(src, artifact_dir / name)
for name in ["val_predictions.json", "val_score.json"]:
    src = Path(name)
    if src.exists():
        shutil.copy2(src, artifact_dir / name)
print("Artifacts:", sorted(p.name for p in artifact_dir.iterdir()))
sweep_src = Path("./models/checkpoint_sweep")
if sweep_src.exists():
    sweep_dst = artifact_dir / "checkpoint_sweep"
    if sweep_dst.exists():
        shutil.rmtree(sweep_dst)
    shutil.copytree(sweep_src, sweep_dst)
    print("Copied checkpoint_sweep")
